In [ ]:
!pip install tensorflow scikit-learn

## 1. Import Libraries

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import pickle

from sklearn.metrics import confusion_matrix, classification_report
from tensorflow.keras.utils import to_categorical
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv2D, MaxPooling2D, Flatten, Dense, Dropout, BatchNormalization
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
from tensorflow.keras.preprocessing.image import ImageDataGenerator

## 2. Load Dataset

In [ ]:
# Load FER2013 dataset
df = pd.read_csv('fer2013.csv')

# Preview the first 5 rows
display(df.head())

## 3. Exploratory Data Analysis (EDA)

In [ ]:
# Check dataset split according to the original 'Usage' column
print("Dataset distribution by 'Usage':")
print(df['Usage'].value_counts())

In [ ]:
emotion_labels = {
    0: 'Angry', 
    1: 'Disgust', 
    2: 'Fear', 
    3: 'Happy', 
    4: 'Sad', 
    5: 'Surprise', 
    6: 'Neutral'
}
df['emotion_name'] = df['emotion'].map(emotion_labels)

# Calculate exact counts and percentages for each emotion
emotion_counts = df['emotion_name'].value_counts()
emotion_percentages = df['emotion_name'].value_counts(normalize=True) * 100

dist_df = pd.DataFrame({
    'Count': emotion_counts, 
    'Percentage (%)': emotion_percentages
})
print("Emotion Distribution:")
display(dist_df.round(2))

# Plot class distribution
plt.figure(figsize=(10, 5))
ax = sns.barplot(
    x=emotion_percentages.index, 
    y=emotion_percentages.values, 
    hue=emotion_percentages.index, 
    legend=False, 
    palette="viridis"
)
for container in ax.containers:
    ax.bar_label(container, fmt='%.1f%%', padding=3, fontsize=10, fontweight='bold')
plt.title('Emotion Class Distribution in % (Class Imbalance)', fontsize=14, fontweight='bold')
plt.ylabel('Percentage (%)')
plt.xlabel('Emotion')
plt.ylim(0, max(emotion_percentages.values) + 4)
plt.grid(axis='y', linestyle='--', alpha=0.5)
plt.tight_layout()
plt.show()

In [ ]:
# Preview one sample image per emotion class
fig, axes = plt.subplots(1, 7, figsize=(20, 3))

for i, (emotion_num, emotion_str) in enumerate(emotion_labels.items()):
    # Get the first image matching this emotion
    sample = df[df['emotion'] == emotion_num].iloc[0]
    
    # Convert pixel string into a 48x48 numpy array
    pixels = np.fromstring(sample['pixels'], sep=' ').reshape(48, 48)
    
    axes[i].imshow(pixels, cmap='gray')
    axes[i].set_title(emotion_str, fontsize=12, fontweight='bold')
    axes[i].axis('off')

plt.suptitle("Sample Images for Each Emotion Class", fontsize=16, fontweight='bold', y=1.05)
plt.tight_layout()
plt.show()

## 4. Data Preprocessing (Normalization, Reshaping & Encoding)

In [ ]:
# Normalize casing/whitespace on 'Usage' so 'Training', 'PublicTest'/'Publictest', 'PrivateTest'/'Privatetest' all match
usage_norm = df['Usage'].astype(str).str.strip().str.lower()

train_df = df[usage_norm == 'training']
val_df = df[usage_norm == 'publictest']
test_df = df[usage_norm == 'privatetest']

# Function to extract pixels, normalize, and reshape
def preprocess_data(data_subset):
    # Convert pixel strings to numpy arrays
    pixels = data_subset['pixels'].apply(lambda x: np.fromstring(x, sep=' '))
    # Stack and normalize pixel values to [0, 1]
    X = np.vstack(pixels.values) / 255.0
    # Reshape for CNN input (48x48 grayscale, 1 channel)
    X = X.reshape(-1, 48, 48, 1)
    # One-Hot Encode emotion labels
    y = to_categorical(data_subset['emotion'].values, num_classes=7)
    return X, y

# Apply preprocessing
X_train, y_train = preprocess_data(train_df)
X_val, y_val = preprocess_data(val_df)
X_test, y_test = preprocess_data(test_df)

print(f"Training Set   (Training)   : X={X_train.shape}, y={y_train.shape}")
print(f"Validation Set (PublicTest) : X={X_val.shape}, y={y_val.shape}")
print(f"Test Set       (PrivateTest): X={X_test.shape}, y={y_test.shape}")

## 5. Build CNN Architecture

In [ ]:
model = Sequential([
    Conv2D(64, (3, 3), activation='relu', padding='same', input_shape=(48, 48, 1)),
    BatchNormalization(),
    Conv2D(64, (3, 3), activation='relu', padding='same'),
    BatchNormalization(),
    MaxPooling2D(pool_size=(2, 2)),
    Dropout(0.25),

    Conv2D(128, (3, 3), activation='relu', padding='same'),
    BatchNormalization(),
    Conv2D(128, (3, 3), activation='relu', padding='same'),
    BatchNormalization(),
    MaxPooling2D(pool_size=(2, 2)),
    Dropout(0.25),

    Conv2D(256, (3, 3), activation='relu', padding='same'),
    BatchNormalization(),
    MaxPooling2D(pool_size=(2, 2)),
    Dropout(0.25),

    Flatten(),
    Dense(256, activation='relu'),
    BatchNormalization(),
    Dropout(0.5),
    Dense(7, activation='softmax')
])

model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])
model.summary()

## 6. Model Training

In [ ]:
early_stop = EarlyStopping(
    monitor='val_accuracy', 
    patience=7, 
    restore_best_weights=True, 
    verbose=1
)
reduce_lr = ReduceLROnPlateau(
    monitor='val_loss', 
    factor=0.2, 
    patience=3, 
    min_lr=1e-6, 
    verbose=1
)

datagen = ImageDataGenerator(
    rotation_range=10,
    width_shift_range=0.1,
    height_shift_range=0.1,
    zoom_range=0.1,
    horizontal_flip=True
)

epochs = 50
batch_size = 64

print("Starting training...")
history = model.fit(
    datagen.flow(X_train, y_train, batch_size=batch_size),
    validation_data=(X_val, y_val),  # Using PublicTest for validation
    epochs=epochs,
    callbacks=[early_stop, reduce_lr],
    verbose=1
)

## 7. Performance Evaluation & Visualizations

In [ ]:
# Evaluate on the PrivateTest set
test_loss, test_acc = model.evaluate(X_test, y_test, verbose=1)

print("\n=== Final Metrics Summary ===")
print(f"Train Accuracy (last epoch)      : {history.history['accuracy'][-1]*100:.2f}%")
print(f"Validation Accuracy (last epoch) : {history.history['val_accuracy'][-1]*100:.2f}%")
print(f"Best Validation Accuracy         : {max(history.history['val_accuracy'])*100:.2f}%")
print(f"Test Accuracy (PrivateTest)      : {test_acc*100:.2f}%")
print(f"Train Loss (last epoch)          : {history.history['loss'][-1]:.4f}")
print(f"Validation Loss (last epoch)     : {history.history['val_loss'][-1]:.4f}")
print(f"Test Loss (PrivateTest)          : {test_loss:.4f}")

In [ ]:
# Plot Training & Validation Accuracy and Loss Curves
fig, axes = plt.subplots(1, 2, figsize=(15, 5))

# Accuracy Plot
axes[0].plot(history.history['accuracy'], label='Train Accuracy', linewidth=2)
axes[0].plot(history.history['val_accuracy'], label='Validation Accuracy', linewidth=2)
axes[0].set_title('Model Accuracy over Epochs', fontsize=14, fontweight='bold')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Accuracy')
axes[0].legend(loc='lower right')
axes[0].grid(True, linestyle='--', alpha=0.5)

# Loss Plot
axes[1].plot(history.history['loss'], label='Train Loss', linewidth=2)
axes[1].plot(history.history['val_loss'], label='Validation Loss', linewidth=2)
axes[1].set_title('Model Loss over Epochs', fontsize=14, fontweight='bold')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Loss')
axes[1].legend(loc='upper right')
axes[1].grid(True, linestyle='--', alpha=0.5)

plt.tight_layout()
plt.show()

In [ ]:
# Generate predictions on PrivateTest
y_pred_probs = model.predict(X_test)
y_pred = np.argmax(y_pred_probs, axis=1)
y_true = np.argmax(y_test, axis=1)

class_names = [emotion_labels[i] for i in range(7)]

# Print detailed Classification Report
report_dict = classification_report(y_true, y_pred, target_names=class_names, output_dict=True)
print("=== Classification Report (PrivateTest) ===")
print(classification_report(y_true, y_pred, target_names=class_names, digits=4))

# Per-Class F1-Score Bar Chart
f1_scores = [report_dict[c]['f1-score'] * 100 for c in class_names]
plt.figure(figsize=(10, 4))
ax = sns.barplot(x=class_names, y=f1_scores, hue=class_names, legend=False, palette='magma')
for container in ax.containers:
    ax.bar_label(container, fmt='%.1f%%', padding=3, fontsize=10, fontweight='bold')
plt.title('Per-Class F1-Score on PrivateTest (%)', fontsize=14, fontweight='bold')
plt.ylabel('F1-Score (%)')
plt.xlabel('Emotion')
plt.ylim(0, 100)
plt.grid(axis='y', linestyle='--', alpha=0.5)
plt.tight_layout()
plt.show()

# Compute Normalized Confusion Matrix
cm = confusion_matrix(y_true, y_pred)
cm_normalized = cm.astype('float') / cm.sum(axis=1)[:, np.newaxis]

plt.figure(figsize=(9, 7))
sns.heatmap(
    cm_normalized, 
    annot=True, 
    fmt='.2f', 
    cmap='Blues', 
    xticklabels=class_names, 
    yticklabels=class_names,
    cbar_kws={'label': 'Normalized Recall'}
)
plt.title('Normalized Confusion Matrix (PrivateTest)', fontsize=14, fontweight='bold')
plt.ylabel('True Emotion')
plt.xlabel('Predicted Emotion')
plt.tight_layout()
plt.show()

In [ ]:
# Sample Predictions Grid (True vs Predicted with Confidence)
np.random.seed(42)
sample_indices = np.random.choice(len(X_test), size=14, replace=False)

fig, axes = plt.subplots(2, 7, figsize=(20, 6))
axes = axes.flatten()

for idx, sample_idx in enumerate(sample_indices):
    img = X_test[sample_idx].reshape(48, 48)
    true_label = class_names[y_true[sample_idx]]
    pred_label = class_names[y_pred[sample_idx]]
    confidence = y_pred_probs[sample_idx][y_pred[sample_idx]] * 100
    
    color = 'green' if true_label == pred_label else 'red'
    axes[idx].imshow(img, cmap='gray')
    axes[idx].set_title(f"True: {true_label}\nPred: {pred_label} ({confidence:.1f}%)", color=color, fontsize=10, fontweight='bold')
    axes[idx].axis('off')

plt.suptitle('Sample Predictions on PrivateTest (Green = Correct, Red = Misclassified)', fontsize=15, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

## 8. Key Conclusions & Takeaways

- **Dataset Split & Class Imbalance:** The model uses the official FER2013 split (`Training`, `PublicTest` for validation, and `PrivateTest` for final evaluation). Because `Happy` is heavily overrepresented while `Disgust` represents under 2% of samples, per-class recall naturally reflects data availability.
- **Training Dynamics:** Real-time data augmentation (`rotation`, `shift`, `zoom`, `horizontal_flip`) combined with `BatchNormalization`, `Dropout`, `ReduceLROnPlateau`, and `EarlyStopping` prevents overfitting on the 48×48 grayscale inputs and stabilizes convergence.
- **Per-Class Performance:** High-contrast expressions such as `Happy` and `Surprise` achieve the strongest F1-scores, whereas subtle or visually overlapping negative emotions (`Fear`, `Sad`, `Angry`) exhibit higher cross-confusion in the normalized confusion matrix.
- **Deployment Readiness:** Exporting the trained weights (`moodlens_cnn_model.h5`) enables lightweight, client-side real-time inference in the MoodLens web interface.

## 9. Save Trained Model

In [ ]:
# Save the model for web export (tools/export_weights.py)
model.save('moodlens_cnn_model.h5')
print("Model saved successfully as 'moodlens_cnn_model.h5'")